In [ ]:
# RSNA Knee - self-contained frozen-model SUBMISSION (Kaggle T4, internet OFF).
# Embeds train+test with frozen DINOv2-S, trains an attention-MIL head on soft
# rule-x-Qwen labels (bundled), predicts test -> submission.csv. No external steps.
import os, glob, time, numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel
import pydicom
K = 8; IMG = 224
LABELS = ['ACL','MCL','Medial Meniscus','Lateral Meniscus','Medial OA','Lateral OA','PF OA',
          'Effusion','Synovitis',"Baker's",'Contusion','Fracture']
print('torch', torch.__version__, '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

base = None
for r in sorted(glob.glob('/kaggle/input/*')):
    h = glob.glob(r+'/**/test_series.csv', recursive=True)
    if h: base = os.path.dirname(h[0]); break
print('COMP base:', base)
test_series = pd.read_csv(os.path.join(base,'test_series.csv'))
train_series = pd.read_csv(os.path.join(base,'train_series.csv'))
sub = pd.read_csv(os.path.join(base,'sample_submission.csv'))
soft = pd.read_csv(glob.glob('/kaggle/input/**/pseudo_labels_v1.csv', recursive=True)[0]).set_index('StudyInstanceUID')
print('test studies', sub.shape[0], '| train soft labels', soft.shape)


In [ ]:
mdir = None
for c in glob.glob('/kaggle/input/**/config.json', recursive=True):
    d = os.path.dirname(c)
    if glob.glob(d+'/pytorch_model.bin') or glob.glob(d+'/*.safetensors'): mdir=d; break
enc = AutoModel.from_pretrained(mdir).eval().half().cuda()
print('encoder loaded', mdir)
IM_MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1); IM_STD = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1)

def paths_sorted(sd):
    fs = glob.glob(sd+'/*.dcm'); h=[]
    for f in fs:
        try: h.append((float(getattr(pydicom.dcmread(f, stop_before_pixels=True),'InstanceNumber',0) or 0), f))
        except Exception: h.append((0.0,f))
    h.sort(); return [f for _,f in h]
def samp(n,k):
    if n<=k: return list(range(n))
    lo,hi=int(n*.12),int(n*.88);
    if hi<=lo: lo,hi=0,n
    return [int(round(x)) for x in np.linspace(lo,hi-1,k)]
def dec(p):
    d=pydicom.dcmread(p); a=d.pixel_array.astype(np.float32)
    return a*float(getattr(d,'RescaleSlope',1) or 1)+float(getattr(d,'RescaleIntercept',0) or 0)
def nr(a):
    lo,hi=np.percentile(a,0.5),np.percentile(a,99.5); a=np.clip((a-lo)/max(hi-lo,1e-6),0,1)
    return F.interpolate(torch.from_numpy(a)[None,None].float(),size=(IMG,IMG),mode='bilinear',align_corners=False)[0,0]
@torch.no_grad()
def embed_series(paths):
    idx=samp(len(paths),K); cache={}
    def g(i):
        i=max(0,min(len(paths)-1,i))
        if i not in cache:
            try: cache[i]=nr(dec(paths[i]))
            except Exception: cache[i]=torch.zeros(IMG,IMG)
        return cache[i]
    x=torch.stack([torch.stack([g(i-1),g(i),g(i+1)]) for i in idx]); x=(x-IM_MEAN)/IM_STD
    return enc(pixel_values=x.half().cuda(), interpolate_pos_encoding=True).last_hidden_state[:,0].float().cpu().numpy()
PL={'Sagittal':0,'Coronal':1,'Axial':2}

def embed_study(study, series_df):
    rows=[]
    for _,sr in series_df[series_df['StudyInstanceUID']==study].iterrows():
        sd=os.path.join(base, 'train_series' if 'train' in series_df.attrs.get('kind','') else 'test_series', study, sr['SeriesInstanceUID'])
        if not os.path.isdir(sd):
            alt=glob.glob(os.path.join(base,'**',sr['SeriesInstanceUID']),recursive=True); sd=alt[0] if alt else None
        if not sd: continue
        p=paths_sorted(sd)
        if not p: continue
        z=embed_series(p); pid=PL.get(sr['Anatomical_Plane'],0); fb=int(sr['Fluid_Sensitive'])
        for j in range(z.shape[0]): rows.append((z[j],pid,fb))
    return rows
print('helpers ready')


In [ ]:
# ---- embed TRAIN ----
train_series.attrs['kind']='train'
train_studies=[s for s in soft.index if s in set(train_series['StudyInstanceUID'])]
t0=time.time(); TR={}
for i,st in enumerate(train_studies):
    r=embed_study(st, train_series)
    if r: TR[st]=r
    if i%400==0: print(f'  train {i}/{len(train_studies)} {time.time()-t0:.0f}s', flush=True)
print('train embedded', len(TR), f'{time.time()-t0:.0f}s')


In [ ]:
# ---- MIL head + train on soft labels ----
class MIL(nn.Module):
    def __init__(s,di=384,d=256,n=12):
        super().__init__(); s.proj=nn.Sequential(nn.Linear(di,d),nn.LayerNorm(d),nn.GELU())
        s.plane=nn.Embedding(3,d); s.fluid=nn.Embedding(2,d)
        s.V=nn.Linear(d,128); s.U=nn.Linear(d,128); s.w=nn.Linear(128,1)
        s.head=nn.Sequential(nn.Dropout(0.3),nn.Linear(d,n))
    def forward(s,x,pid,fb):
        h=s.proj(x)+s.plane(pid)+s.fluid(fb)
        a=torch.softmax(s.w(torch.tanh(s.V(h))*torch.sigmoid(s.U(h))),0)
        return s.head((a*h).sum(0))
dev='cuda'
def to_t(rows):
    x=torch.tensor(np.stack([r[0] for r in rows]),dtype=torch.float32,device=dev)
    pid=torch.tensor([r[1] for r in rows],device=dev); fb=torch.tensor([r[2] for r in rows],device=dev)
    return x,pid,fb
torch.manual_seed(0)
models=[]
for seed in range(3):
    torch.manual_seed(seed); m=MIL().to(dev); opt=torch.optim.AdamW(m.parameters(),lr=1e-3,weight_decay=1e-2)
    ids=list(TR.keys())
    for ep in range(30):
        m.train(); np.random.shuffle(ids)
        for st in ids:
            x,pid,fb=to_t(TR[st]); y=torch.tensor(soft.loc[st,LABELS].values.astype(np.float32),device=dev)
            loss=F.binary_cross_entropy_with_logits(m(x,pid,fb),y)
            opt.zero_grad(); loss.backward(); opt.step()
    models.append(m.eval()); print(f'  seed {seed} trained')
print('heads trained', len(models))


In [ ]:
# ---- embed TEST + predict + write submission ----
test_series.attrs['kind']='test'
test_ids=list(sub['StudyInstanceUID']); t0=time.time(); preds={}
for i,st in enumerate(test_ids):
    rows=embed_study(st, test_series)
    if not rows:
        preds[st]=np.full(12,0.5); continue
    x,pid,fb=to_t(rows)
    with torch.no_grad():
        ps=np.mean([torch.sigmoid(m(x,pid,fb)).cpu().numpy() for m in models],0)
    preds[st]=ps
    if i%200==0: print(f'  test {i}/{len(test_ids)} {time.time()-t0:.0f}s', flush=True)
out=pd.DataFrame({'StudyInstanceUID':test_ids})
for j,l in enumerate(LABELS): out[l]=[preds[s][j] for s in test_ids]
out=out[['StudyInstanceUID']+LABELS]
out.to_csv('submission.csv',index=False)
print('WROTE submission.csv', out.shape, f'{time.time()-t0:.0f}s'); print(out.head())
